# ResNet50 on the NIH ChestX-ray14 official split

This notebook trains **one CNN model only: ResNet50** for the 14-label NIH ChestX-ray14 multilabel classification task.

Attach the NIH dataset before running. The notebook discovers `Data_Entry_2017.csv`, image files, `train_val_list.txt`, and `test_list.txt` below `DATA_ROOT`. It uses the official image lists and creates validation from `train_val_list.txt` at the patient level with seed 42.

In [ ]:
# Configuration
import os, glob, json, hashlib, random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from torchvision import transforms, models
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
SEED = 42
EXPECTED_COUNTS = {'train_val': 86524, 'test': 25596, 'total': 112120}
EXPECTED_LIST_SHA256 = {
    'train_val_list.txt': '61fbe896321c1c1c8b75f3e4f3a08e4fef6486d95ef8a667c31d4d60dca6cb81',
    'test_list.txt': '38ca5ef7f756092946f57c1a59faca882ed589a1ab1f72590b45dc06c6d5e1cc'}
DATA_ROOT = os.environ.get('NIH_DATA_ROOT', '/kaggle/input/nih-chest-xrays/data')
OUT_DIR = os.environ.get('NIH_OUT_DIR', '/kaggle/working/single_model/output/resnet50_nih_official')
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = min(4, os.cpu_count() or 1)
EPOCHS = 12
PATIENCE = 3
LR = 1e-4
WEIGHT_DECAY = 1e-4
USE_IMAGENET_WEIGHTS = True
DEBUG = False  # set True only for a quick smoke test

CLASSES = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia',
           'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']
NUM_CLASSES = len(CLASSES)
os.makedirs(OUT_DIR, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
print('data root:', DATA_ROOT)

## 1. Load metadata and reproduce the official split

In [ ]:
def locate(filename):
    direct = os.path.join(DATA_ROOT, filename)
    if os.path.exists(direct): return direct
    matches = list(Path(DATA_ROOT).rglob(filename))
    if not matches: raise FileNotFoundError(f'{filename} not found below {DATA_ROOT}')
    return str(matches[0])

meta_path = locate('Data_Entry_2017.csv')
train_val_path = locate('train_val_list.txt')
test_path = locate('test_list.txt')
meta = pd.read_csv(meta_path)

# Find all PNGs without assuming the Kaggle folder layout.
image_paths = {}
for p in Path(DATA_ROOT).rglob('*.png'):
    image_paths[p.name] = str(p)
meta['path'] = meta['Image Index'].map(image_paths)
assert meta['path'].notna().all(), f"Missing images: {meta['path'].isna().sum()}"

for c in CLASSES:
    meta[c] = meta['Finding Labels'].str.split('|').map(lambda labels: float(c in labels))
meta = meta.sort_values('Image Index').reset_index(drop=True)
train_val_ids = set(Path(train_val_path).read_text().split())
test_ids = set(Path(test_path).read_text().split())
def list_sha(ids): return hashlib.sha256('\n'.join(sorted(ids)).encode()).hexdigest()
assert len(meta) == EXPECTED_COUNTS['total']
assert len(train_val_ids) == EXPECTED_COUNTS['train_val'] and len(test_ids) == EXPECTED_COUNTS['test']
assert list_sha(train_val_ids) == EXPECTED_LIST_SHA256['train_val_list.txt']
assert list_sha(test_ids) == EXPECTED_LIST_SHA256['test_list.txt']
assert not train_val_ids & test_ids
assert train_val_ids | test_ids == set(meta['Image Index'])

meta['split'] = np.where(meta['Image Index'].isin(test_ids), 'test', 'trainval')
trainval = meta[meta['split'].eq('trainval')]
_, val_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED).split(
    trainval, groups=trainval['Patient ID']))
meta.loc[trainval.index, 'split'] = 'train'
meta.loc[trainval.index[val_idx], 'split'] = 'val'

if DEBUG:
    meta = pd.concat([meta[meta.split.eq(s)].sample(min(n, (meta.split == s).sum()), random_state=SEED)
                     for s, n in [('train', 512), ('val', 128), ('test', 128)]])

df_train = meta[meta.split.eq('train')].reset_index(drop=True)
df_val = meta[meta.split.eq('val')].reset_index(drop=True)
df_test = meta[meta.split.eq('test')].reset_index(drop=True)
for name, frame in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f'{name:5s}: {len(frame):,} images | {frame["Patient ID"].nunique():,} patients')
assert set(df_train['Patient ID']).isdisjoint(df_val['Patient ID'])
assert set(df_train['Patient ID']).isdisjoint(df_test['Patient ID'])
assert set(df_val['Patient ID']).isdisjoint(df_test['Patient ID'])
meta.to_csv(os.path.join(OUT_DIR, 'metadata_with_official_split.csv'), index=False)

## 2. Dataset, transforms, and the single ResNet50 model

In [ ]:
class NIHChestXray(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform: image = self.transform(image)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=np.float32))
        return image, target, row['Image Index']

mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(7),
    transforms.ToTensor(), transforms.Normalize(mean, std)])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(), transforms.Normalize(mean, std)])

loaders = {
    'train': DataLoader(NIHChestXray(df_train, train_transform), batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()),
    'val': DataLoader(NIHChestXray(df_val, eval_transform), batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()),
    'test': DataLoader(NIHChestXray(df_test, eval_transform), batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())}

weights = models.ResNet50_Weights.DEFAULT if USE_IMAGENET_WEIGHTS else None
model = models.resnet50(weights=weights)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.2, patience=1)
print(f'ResNet50 parameters: {sum(p.numel() for p in model.parameters()):,}')

## 3. Train with validation macro AUROC

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    total_loss, n = 0.0, 0
    all_y, all_p = [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, targets, _ in tqdm(loader, leave=False):
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, targets)
            if training:
                optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(images); n += len(images)
            all_y.append(targets.detach().cpu().numpy()); all_p.append(torch.sigmoid(logits).detach().cpu().numpy())
    y, p = np.concatenate(all_y), np.concatenate(all_p)
    aucs = [roc_auc_score(y[:, i], p[:, i]) for i in range(NUM_CLASSES) if len(np.unique(y[:, i])) > 1]
    return total_loss / n, float(np.mean(aucs)), y, p

history, best_auc, bad_epochs = [], -np.inf, 0
best_path = os.path.join(OUT_DIR, 'resnet50_best.pt')
for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss, train_auc, _, _ = run_epoch(loaders['train'], True)
    val_loss, val_auc, _, _ = run_epoch(loaders['val'], False)
    scheduler.step(val_auc)
    record = {'epoch': epoch, 'train_loss': train_loss, 'train_auc': train_auc, 'val_loss': val_loss, 'val_auc': val_auc}
    history.append(record)
    print(f"epoch {epoch:02d} | train loss {train_loss:.4f}, AUROC {train_auc:.4f} | val loss {val_loss:.4f}, AUROC {val_auc:.4f} | {time.time()-start:.0f}s")
    if val_auc > best_auc:
        best_auc, bad_epochs = val_auc, 0
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch, 'val_auc': val_auc, 'classes': CLASSES}, best_path)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE: print('Early stopping'); break
pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, 'training_history.csv'), index=False)
print('best checkpoint:', best_path)

## 4. Test evaluation and exported predictions

In [ ]:
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
test_loss, test_auc, y_test, p_test = run_epoch(loaders['test'], False)
per_class = []
for i, name in enumerate(CLASSES):
    per_class.append({'class': name, 'auroc': roc_auc_score(y_test[:, i], p_test[:, i]),
                      'auprc': average_precision_score(y_test[:, i], p_test[:, i]),
                      'prevalence': y_test[:, i].mean()})
metrics = pd.DataFrame(per_class)
metrics.to_csv(os.path.join(OUT_DIR, 'test_per_class_metrics.csv'), index=False)
np.savez_compressed(os.path.join(OUT_DIR, 'test_predictions.npz'), y_true=y_test, y_score=p_test, classes=CLASSES)
summary = {'model': 'resnet50', 'dataset': 'NIH_CXR14', 'split': 'NIH_OFFICIAL_PV', 'seed': SEED,
           'test_loss': test_loss, 'test_macro_auroc': test_auc, 'best_val_macro_auroc': best_auc,
           'n_test': len(df_test), 'checkpoint': best_path}
with open(os.path.join(OUT_DIR, 'summary.json'), 'w') as f: json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
display(metrics.sort_values('auroc', ascending=False))